# Notebook 03: Vehicle Re-ID Training
**Multi-Camera Tracking System — Kaggle Training Pipeline**

Train vehicle re-identification models on VeRi-776 dataset using torchreid.

**Models**: OSNet-x1.0 (512-dim) + ResNet50-IBN-a (2048-dim)  
**Runtime**: GPU T4/P100 | **Duration**: ~2-3 hours total

## Target Metrics
| Model | mAP | Rank-1 |
|-------|-----|--------|
| OSNet-x1.0 | ≥78% | ≥95% |
| ResNet50-IBN-a | ≥78% | ≥95% |

In [ ]:
!pip install -q torchreid gdown matplotlib pandas

## 1. Setup

In [ ]:
import os
import sys
import json
import time
import shutil
from pathlib import Path
from datetime import datetime

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working/reid_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "dataset": "veri776",
    "veri776_root": "/kaggle/input/veri-776",
    "batch_size_train": 64,
    "batch_size_test": 128,
    "num_workers": 2,
    "height": 224,  # Square crop for vehicles
    "width": 224,
}

print(f"Device: {DEVICE}")

## 2. Dataset Registration

In [ ]:
import torchreid

# VeRi-776 dataset setup
# torchreid supports VeRi natively
datamanager = torchreid.data.ImageDataManager(
    root=CONFIG["veri776_root"],
    sources=["veri"],
    targets=["veri"],
    height=CONFIG["height"],
    width=CONFIG["width"],
    batch_size_train=CONFIG["batch_size_train"],
    batch_size_test=CONFIG["batch_size_test"],
    transforms=["random_flip", "random_crop", "random_erasing", "color_jitter"],
    num_workers=CONFIG["num_workers"],
)

print(f"Train: {datamanager.num_train_pids} vehicle IDs, {datamanager.num_train_cams} cameras")

## 3. Model A: ResNet50-IBN-a Training

**Config**: 120 epochs, Adam lr=3.5e-4, cosine LR schedule, triplet + cross-entropy + label smoothing

Note: We use 224×224 input for vehicles (vs 256×128 for persons) since vehicles are more square-shaped.

In [ ]:
model_resnet = torchreid.models.build_model(
    name="resnet50_ibn_a",
    num_classes=datamanager.num_train_pids,
    loss="softmax",
    pretrained=True,
)
model_resnet = model_resnet.to(DEVICE)

optimizer_resnet = torchreid.optim.build_optimizer(
    model_resnet, optim="adam", lr=3.5e-4, weight_decay=5e-4,
)
scheduler_resnet = torchreid.optim.build_lr_scheduler(
    optimizer_resnet, lr_scheduler="cosine", max_epoch=120,
)

print(f"Parameters: {sum(p.numel() for p in model_resnet.parameters()):,}")

In [ ]:
engine_resnet = torchreid.engine.ImageSoftmaxEngine(
    datamanager, model_resnet,
    optimizer=optimizer_resnet, scheduler=scheduler_resnet,
    label_smooth=True,
)

print("=" * 60)
print("Training ResNet50-IBN-a on VeRi-776")
print("=" * 60)

start_time = time.time()
engine_resnet.run(
    save_dir=str(OUTPUT_DIR / "vehicle_resnet50_ibn_a"),
    max_epoch=120, eval_freq=20, print_freq=50,
    test_only=False, fixbase_epoch=10,
)
resnet_time = time.time() - start_time
print(f"\nCompleted in {resnet_time / 3600:.1f} hours")

## 4. Evaluate ResNet50-IBN-a

In [ ]:
resnet_results = engine_resnet.run(
    save_dir=str(OUTPUT_DIR / "vehicle_resnet50_ibn_a"),
    max_epoch=120, test_only=True,
)
print("\nResNet50-IBN-a (Vehicle) Results:")
print(f"  mAP:    {resnet_results.get('mAP', 'N/A')}")
print(f"  Rank-1: {resnet_results.get('Rank-1', 'N/A')}")

## 5. Model B: OSNet-x1.0 Training

**Config**: 150 epochs, Adam lr=3.5e-4, cosine LR, label smoothing

In [ ]:
model_osnet = torchreid.models.build_model(
    name="osnet_x1_0",
    num_classes=datamanager.num_train_pids,
    loss="softmax",
    pretrained=True,
)
model_osnet = model_osnet.to(DEVICE)

optimizer_osnet = torchreid.optim.build_optimizer(
    model_osnet, optim="adam", lr=3.5e-4, weight_decay=5e-4,
)
scheduler_osnet = torchreid.optim.build_lr_scheduler(
    optimizer_osnet, lr_scheduler="cosine", max_epoch=150,
)

print(f"Parameters: {sum(p.numel() for p in model_osnet.parameters()):,}")

In [ ]:
engine_osnet = torchreid.engine.ImageSoftmaxEngine(
    datamanager, model_osnet,
    optimizer=optimizer_osnet, scheduler=scheduler_osnet,
    label_smooth=True,
)

print("=" * 60)
print("Training OSNet-x1.0 on VeRi-776")
print("=" * 60)

start_time = time.time()
engine_osnet.run(
    save_dir=str(OUTPUT_DIR / "vehicle_osnet_x1_0"),
    max_epoch=150, eval_freq=20, print_freq=50,
    test_only=False, fixbase_epoch=10,
)
osnet_time = time.time() - start_time
print(f"\nCompleted in {osnet_time / 3600:.1f} hours")

## 6. Evaluate OSNet-x1.0

In [ ]:
osnet_results = engine_osnet.run(
    save_dir=str(OUTPUT_DIR / "vehicle_osnet_x1_0"),
    max_epoch=150, test_only=True,
)
print("\nOSNet-x1.0 (Vehicle) Results:")
print(f"  mAP:    {osnet_results.get('mAP', 'N/A')}")
print(f"  Rank-1: {osnet_results.get('Rank-1', 'N/A')}")

## 7. Results Comparison

In [ ]:
results_summary = {
    "resnet50_ibn_a": {
        "embedding_dim": 2048,
        "input_size": [224, 224],
        "train_time_hours": resnet_time / 3600,
        "epochs": 120,
    },
    "osnet_x1_0": {
        "embedding_dim": 512,
        "input_size": [224, 224],
        "train_time_hours": osnet_time / 3600,
        "epochs": 150,
    },
}

results_path = OUTPUT_DIR / "vehicle_reid_results.json"
with open(results_path, "w") as f:
    json.dump(results_summary, f, indent=2)

print("Vehicle ReID Training Summary")
print("=" * 60)
print(f"{'Model':<20} {'Dim':>5} {'Epochs':>7} {'Time (h)':>9}")
print("-" * 60)
for name, info in results_summary.items():
    print(f"{name:<20} {info['embedding_dim']:>5} {info['epochs']:>7} {info['train_time_hours']:>9.1f}")

## 8. Export Models

In [ ]:
export_dir = Path("/kaggle/working/exported_models")
export_dir.mkdir(parents=True, exist_ok=True)

models_to_export = {
    "vehicle_resnet50_ibn_a_veri776.pth": OUTPUT_DIR / "vehicle_resnet50_ibn_a" / "model" / "model-best.pth.tar",
    "vehicle_osnet_x1_0_veri776.pth": OUTPUT_DIR / "vehicle_osnet_x1_0" / "model" / "model-best.pth.tar",
}

for export_name, src_path in models_to_export.items():
    if src_path.exists():
        dst = export_dir / export_name
        shutil.copy2(src_path, dst)
        print(f"Exported: {export_name} ({dst.stat().st_size / 1024**2:.1f} MB)")
    else:
        alt = src_path.parent / f"model.pth.tar-{120 if 'resnet' in export_name else 150}"
        if alt.exists():
            shutil.copy2(alt, export_dir / export_name)
            print(f"Exported (alt): {export_name}")
        else:
            print(f"Warning: {src_path} not found")

metadata = {
    "task": "vehicle_reid",
    "dataset": "veri776",
    "models": {
        "resnet50_ibn_a": {
            "architecture": "resnet50_ibn_a",
            "embedding_dim": 2048,
            "input_size": [224, 224],
            "file": "vehicle_resnet50_ibn_a_veri776.pth",
        },
        "osnet_x1_0": {
            "architecture": "osnet_x1_0",
            "embedding_dim": 512,
            "input_size": [224, 224],
            "file": "vehicle_osnet_x1_0_veri776.pth",
        },
    },
    "trained_at": datetime.now().isoformat(),
}

with open(export_dir / "vehicle_reid_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nDownload from: {export_dir}")
print("Place in local project: models/reid/")

## Next Steps
1. **Download** exported models from `/kaggle/working/exported_models/`
2. Place in local project: `models/reid/`
3. Both person (Notebook 02) and vehicle models are now ready
4. Optionally proceed to **Notebook 04** for TransReID (stretch goal)

### Models Summary
After running Notebooks 02 and 03, you should have:
- `person_osnet_x1_0_market1501.pth` — Person ReID (512-dim)
- `person_resnet50_ibn_a_market1501.pth` — Person ReID (2048-dim)
- `vehicle_osnet_x1_0_veri776.pth` — Vehicle ReID (512-dim)
- `vehicle_resnet50_ibn_a_veri776.pth` — Vehicle ReID (2048-dim)